# Python-to-Rust Performance Lab

## Introduction

This reference notebook explores one workflow for comparing a Python implementation with an LLM-generated Rust port. It collects local toolchain information, sends a Python benchmark to a selected model, writes the returned source to `main.rs`, compiles it with `rustc`, and presents both runs in a Gradio interface.

### Before you run it

- Configure the API key for the provider you intend to use in your environment or `.env` file. The setup cell reports whether keys are present; it does not validate them.
- Run the notebook from its `Week 4` directory so `system_info.py`, `styles.py`, `main.rs`, and the compiled `main` executable resolve as expected.
- The notebook executes Python with `exec` and runs generated native code as your local user. Use only code you trust; this is a demonstration, not a sandbox.
- The UI and compiler command are configured for Rust. Although one cell mentions C++, switching `language` alone does not configure a C++ compiler or runner.

The notebook is organized into provider setup, compiler configuration, translation and execution helpers, and a benchmark UI. Each section summary explains the role of its cells and how to interpret the comparison.

In [1]:
# Imports
import os
import io
import sys
import shutil
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess as sub
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')    


api_keys = {
    'OpenAI': openai_api_key,
    'Anthropic': anthropic_api_key,
    'Google': google_api_key,
    'Grok': grok_api_key,
    'Groq': groq_api_key,
    'OpenRouter': openrouter_api_key,
}
for provider, api_key in api_keys.items():
    status = 'found and ready' if api_key else 'not found'
    print(f'{provider} API key {status}.')

OpenAI API key found and ready.
Anthropic API key found and ready.
Google API key found and ready.
Grok API key found and ready.
Groq API key found and ready.
OpenRouter API key found and ready.


In [3]:
# connect to libraries

openai = OpenAI()

anthropic_url = 'https://api.anthropic.com/v1'
gemini_url = 'https://generativelanguage.googleapis.com/v1beta/openai'
grok_url = 'https://api.x.ai/v1'
groq_url = 'https://api.groq.com/openai/v1'
ollama_url = 'http://localhost:11434/v1'
openrouter_url = 'https://openrouter.ai/api/v1'

anthropic = OpenAI(api_key=anthropic_api_key,
                   base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key,
                base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key,
              base_url=grok_url)
groq = OpenAI(api_key=groq_api_key,
              base_url=groq_url)
ollama = OpenAI(api_key='ollama',
                base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key,
                    base_url=openrouter_url)

In [4]:
models= [
    'gpt-6.1-sol',
    'claude-sonnet-5-5',
    'grok-4.3',
    'gemini-4-argon',
    'qwen2.5-coder',
    'deepseek-coder-v2',
    'gpt-oss:20b',
    'qwen/qwen3-coder-30b-a3b-instruct',
    'openai/gpt-oss-120b'
]

clients = {
    'gpt-6.1-sol': openai,
    'claude-sonnet-5-5': anthropic,
    'grok-4.3': grok,
    'gemini-4-argon': gemini,
    'openai/gpt-oss-120b': groq,
    'qwen2.5-coder': ollama,
    'deepseek-coder-v2': ollama,
    'gpt-oss:20b': ollama,
    'qwen/qwen3-coder-30b-a3b-instruct': openrouter,
}

In [5]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': '/Users/michaelperrine/.cargo/bin/rustc',
  'version': 'rustc 1.96.1 (31fca3adb 2026-06-26)',
  'host_triple': 'aarch64-apple-darwin',
  'release': '1.96.1',
  'commit_hash': '31fca3adb283cc9dfd56b49cdee9a96eb9c96ffd'},
 'cargo': {'path': '/Users/michaelperrine/.cargo/bin/cargo',
  'version': 'cargo 1.96.1 (356927216 2026-06-26)'},
 'rustup': {'path': '/Users/michaelperrine/.cargo/bin/rustup',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': 'stable-aarch64-apple-darwin (default)',
  'default_toolchain': '',
  'toolchains': ['stable-aarch64-apple-darwin (active, default)'],
  'targets_installed': ['aarch64-apple-darwin']},
 'rust_analyzer': {'path': '/Users/michaelperrine/.cargo/bin/rust-analyzer'},
 'env': {'CARGO_HOME': '/Users/michaelperrine/.cargo',
  'RUSTUP_HOME': '/Users/michaelperrine/.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"/Users/michaelperrine/.cargo/bin/cargo" build',

### Setup summary

The first cells read provider keys from the environment, create OpenAI-compatible clients for several hosted and local services, and map each selectable model name to its client. The system-information helpers supply platform and Rust details to the model prompt. A key being reported as present does not prove that it is valid or that the provider/model endpoint is reachable.

In [6]:
message = f""" 
Here is a report of the system information for my computer. 
I want to run a Rust compiler to compile a singe rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command + # something here - to achieve the fastest possible performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = openai.chat.completions.create(
    model=models[0],
    messages=[
        {
            'role': 'user',
            'content': message
        }
    ])
display(Markdown(response.choices[0].message.content))

**No installation is needed.** You already have native Apple Silicon Rust (`aarch64-apple-darwin`) and Apple’s compiler/linker tools installed.

For a standalone `main.rs` using only Rust’s standard library, run this Python code from the directory containing the file:

```python
import subprocess
from pathlib import Path

executable = str(Path("main").resolve())

compile_command = [
    "/Users/michaelperrine/.cargo/bin/rustc",
    "main.rs",
    "--edition=2024",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-o", executable,
]

compile_result = subprocess.run(
    compile_command, check=True, text=True, capture_output=True
)

run_command = [executable]

run_result = subprocess.run(
    run_command, check=True, text=True, capture_output=True
)

print(run_result.stdout, end="")
# Inside a function, use: return run_result.stdout
```

These settings enable aggressive optimization, tuning for your **M4 Pro**, full link-time optimization, and a single code-generation unit. `panic=abort` terminates the program immediately on a panic instead of unwinding; omit it if you need panic recovery.

This is a strong runtime-performance configuration, but no compiler flags guarantee the fastest result for every program. Further gains may require benchmarking and profile-guided optimization. The resulting executable is tuned for this computer and may not run on older CPUs.

In [7]:
rustc_path = shutil.which('rustc')
if rustc_path is None:
    raise RuntimeError('rustc was not found on PATH. Install Rust or add its bin directory to PATH.')

compile_command = [
    rustc_path,
    'main.rs',
    '--edition=2024',
    '-C', 'opt-level=3',
    '-C', 'target-cpu=native',
    '-C', 'codegen-units=1',
    '-C', 'lto=fat',
    '-C', 'panic=abort',
    '-C', 'strip=symbols',
    '-o', 'main'

]

run_command = ['./main']

### Compiler setup summary

`compile_command` compiles `main.rs` as a standalone Rust program and places the executable at `./main`. The optimization flags favor runtime speed on the current machine; in particular, `target-cpu=native` makes the result hardware-specific, and `panic=abort` disables panic unwinding. These choices do not establish that the generated program is correct or fastest for every workload. The command is Rust-specific even though the language cell mentions C++.

In [8]:
language = 'Rust' # or "C++"
extension = 'rs' if language == 'Rust' else 'cpp'

system_prompt = f""" 
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f""" 
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main. {language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```

"""

In [9]:
def messages_for(python):
    return [
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': user_prompt_for(python)
        }
    ]

In [10]:
def write_output(code):
    with open(f'main.{extension}', 'w') as f:
        f.write(code)

In [11]:
def port(model, python):
    client = clients[model]
    reasoning_effort = 'high' if model == 'gpt-6.1-sol' else None
    response = client.chat.completions.create(
        model=model,
        messages=messages_for(python),
        reasoning_effort=reasoning_effort
    )
    reply = response.choices[0].message.content
    if not reply:
        raise ValueError(f'{model} returned an empty response.')
    reply = reply.replace('```cpp', '').replace('```rust', '').replace('```', '').strip()
    return reply

In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

In [13]:
# Use the commands from GPT-6.1 Sol

def compile_and_run(code):
    write_output(code)
    try:
        sub.run(compile_command, 
                       check=True,
                       text=True,
                       capture_output=True)
        run_result = sub.run(run_command,
                                    check=True,
                                    text=True,
                                    capture_output=True)
        return run_result.stdout

    except sub.CalledProcessError as e:
        return f"an error occurred \n {e.stderr}"

### Translation and execution summary

`port` builds the model request, selects the corresponding client, and removes common Markdown code fences from the response. `write_output` saves that source, while `compile_and_run` invokes the compiler and executable and returns their standard output. `run_python` captures Python standard output. These helpers do not automatically compare the two outputs; compare them before treating a timing result as meaningful. The Python `exec` and native executable run with the notebook user's permissions, so run only trusted code. Compiler and program nonzero exits are displayed from standard error; other process-launch errors may propagate.

In [14]:
python_hard = """ 
# Be Careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000               # Number of random numbers
initial_seed = 42       # Initial seed for the LCG
min_val = -10           # Minimum value of random numbers
max_val = 10            # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [15]:
from styles import CSS

with gr.Blocks(
        css=CSS, 
        theme=gr.themes.Monochrome(),
        title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label='Python (original)',
                value=python_hard,
                language='python',
                lines=26)
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp" if language == "C++" else None,
                lines=26)
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button('Run Python', elem_classes=['run-btn', 'py'])
        model = gr.Dropdown(models, 
                            value=models[0],
                            show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=['convert-btn'])
        cpp_run = gr.Button(f"Run {language}", elem_classes=['run-btn', 'cpp'])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label='Python result',
                                     lines=8,
                                     elem_classes=['py-out'])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result",
                                  lines=8,
                                  elem_classes=['cpp-out'])
    convert.click(fn=port,
                  inputs=[model, python],
                  outputs=[cpp])
    python_run.click(fn=run_python,
                     inputs=[python],
                     outputs=[python_out])
    cpp_run.click(fn=compile_and_run,
                  inputs=[cpp],
                  outputs=[cpp_out])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Benchmark and UI summary

The Gradio controls let you run the Python input, request a port from the selected model, and compile/run the generated Rust. The sample benchmark computes maximum subarray sums with nested loops, so its work grows quadratically with `n`; with `n = 10,000` and 20 runs, the Python baseline can take a long time. The notebook displays outputs but does not assert equivalence or collect a repeatable benchmark. The final speed ratio is a fixed example from a prior experiment, not a value calculated from the current UI run.

In [16]:
print(f"In Ed's experiment, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")

In Ed's experiment, the GPT-OSS 120B model outcome is 111,037 times faster than the Python code.


## Conclusion

The notebook demonstrates a complete model-assisted porting loop, from provider selection through native compilation and interactive output. For a reliable comparison, first confirm the Python and Rust outputs match, then time repeated runs under the same conditions and report the measurements actually observed. Treat the hard-coded ratio as historical context only, and remember that this notebook currently supports Rust execution rather than a selectable C++ toolchain.